# Extra 4 - Usar embeddings pre-treinados

No projeto final voce provavelmente vai treinar embeddings numa etapa e
usa-los em outra. Os dois pontos onde a sintaxe costuma travar:

1. **Alinhar** a matriz pre-treinada a ordem do vocabulario do modelo novo.
2. **Carregar** com `from_pretrained` e decidir entre **congelar** e
   **fine-tuning**.

Para ter uma matriz "pre-treinada" de verdade, o notebook treina rapido um
skip-gram num corpus pequeno (mesma ideia do modulo 1) e usa o resultado.

Tente resolver antes de olhar o `_solucoes`.

In [1]:
import json
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

corpus = [
    "the meeting is tomorrow morning",
    "send me the meeting notes please",
    "call me after the meeting",
    "lunch after the meeting tomorrow",
    "bring your notes to the meeting",
    "the project meeting moved to friday",
    "win a free cash prize now",
    "claim your free prize today",
    "you won a free cash award",
    "free entry to win a prize now",
    "urgent claim your free cash prize",
    "call now to claim your reward",
]

## 4.1 Treinar rapido um skip-gram (matriz "pre-treinada")

Reaproveite a receita do modulo 1, so que compacta:

1. `tokens`, `pre_word2idx` (indices a partir de 0, sem PAD), `idx2word`.
2. Pares skip-gram com `window = 2`.
3. Modelo: `nn.Embedding(V, 24)` + `nn.Linear(24, V)`; `forward` = `linear(embedding(x))`.
4. Treine 150 epocas, `CrossEntropyLoss`, `Adam(lr=0.01)`.
5. `pre_matrix = model.embedding.weight.detach().numpy()`  -> formato `(V, 24)`.

In [2]:
# 4.1
tokens = [s.lower().split() for s in corpus]
vocab = sorted({w for s in tokens for w in s})
pre_word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in pre_word2idx.items()}
V = len(pre_word2idx)

window = 2
pairs = []
for s in tokens:
    idxs = [pre_word2idx[w] for w in s]
    for i, alvo in enumerate(idxs):
        for j in range(max(0, i - window), min(len(idxs), i + window + 1)):
            if j != i:
                pairs.append((alvo, idxs[j]))

tgt = torch.tensor([a for a, _ in pairs])
ctx = torch.tensor([c for _, c in pairs])

class SkipGram(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.embedding = nn.Embedding(V, d)
        self.linear = nn.Linear(d, V)
    def forward(self, x):
        return self.linear(self.embedding(x))

model = SkipGram(V, 24)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(150):
    opt.zero_grad()
    loss = loss_fn(model(tgt), ctx)
    loss.backward()
    opt.step()

pre_matrix = model.embedding.weight.detach().numpy()
print("pre_matrix:", pre_matrix.shape, "| V =", V)

pre_matrix: (32, 24) | V = 32

## 4.2 Similaridade e analogia

1. `cos(a, b)` = similaridade de cosseno.
2. `mais_parecidas(palavra, topn=3)` usando `pre_matrix` e `pre_word2idx`.
3. `analogia(a, b, c)` = palavra mais proxima de `vec(b) - vec(a) + vec(c)`
   (excluindo `a`, `b`, `c`). Teste `mais_parecidas("free")` e
   `analogia("meeting", "notes", "free")` — com corpus minusculo o resultado
   e instavel, a ideia e so exercitar a conta.

In [3]:
# 4.2
def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def mais_parecidas(palavra, topn=3):
    v = pre_matrix[pre_word2idx[palavra]]
    sims = [(w, cos(v, pre_matrix[i])) for w, i in pre_word2idx.items() if w != palavra]
    sims.sort(key=lambda t: t[1], reverse=True)
    return sims[:topn]

def analogia(a, b, c):
    alvo = pre_matrix[pre_word2idx[b]] - pre_matrix[pre_word2idx[a]] + pre_matrix[pre_word2idx[c]]
    sims = [(w, cos(alvo, pre_matrix[i])) for w, i in pre_word2idx.items()
            if w not in (a, b, c)]
    return max(sims, key=lambda t: t[1])

print("free ->", mais_parecidas("free"))
print("meeting : notes :: free : ", analogia("meeting", "notes", "free"))

free -> [('now', 0.35633790493011475), ('prize', 0.3061354458332062), ('urgent', 0.3058856427669525)]
meeting : notes :: free :  ('urgent', 0.5058879852294922)

## 4.3 Alinhar a matriz a um vocabulario novo

O modelo novo tem o proprio vocabulario, em outra ordem e com palavras que
podem nao estar na matriz pre-treinada.

Dado o `novo_vocab` abaixo (com `<PAD>` e `<UNK>`), monte
`aligned` de formato `(len(novo_vocab), 24)`:

- linha `0` (`<PAD>`): zeros
- para cada palavra: se estiver em `pre_word2idx`, copie a linha correspondente
- se nao estiver (inclui `<UNK>`): vetor aleatorio pequeno
  (`np.random.normal(0, 0.1, 24)`)

Conte quantas palavras foram encontradas (hits) e quantas nao (misses).

In [4]:
# 4.3
novo_vocab = ["<PAD>", "<UNK>", "meeting", "notes", "free", "cash",
              "prize", "call", "email", "deadline", "tomorrow"]

rng = np.random.default_rng(0)
d = pre_matrix.shape[1]
aligned = np.zeros((len(novo_vocab), d), dtype=np.float32)

hits = misses = 0
for i, w in enumerate(novo_vocab):
    if w == "<PAD>":
        continue
    if w in pre_word2idx:
        aligned[i] = pre_matrix[pre_word2idx[w]]
        hits += 1
    else:
        aligned[i] = rng.normal(0, 0.1, d)
        misses += 1

print(f"hits={hits} | misses={misses}")
print("linha <PAD> e zero:", not aligned[0].any())

hits=7 | misses=3
linha <PAD> e zero: True

## 4.4 Carregar no `nn.Embedding` (congelado)

1. `emb_frozen = nn.Embedding.from_pretrained(torch.tensor(aligned),
   freeze=True, padding_idx=0)`.
2. Imprima `emb_frozen.weight.requires_grad` (deve ser `False`).
3. Confirme que `emb_frozen(torch.tensor([[2, 3, 4]]))` bate com
   `aligned[[2, 3, 4]]`.

In [5]:
# 4.4
emb_frozen = nn.Embedding.from_pretrained(
    torch.tensor(aligned), freeze=True, padding_idx=0)

print("requires_grad:", emb_frozen.weight.requires_grad)
saida = emb_frozen(torch.tensor([[2, 3, 4]]))
print("bate com aligned:",
      torch.allclose(saida, torch.tensor(aligned[[2, 3, 4]]).unsqueeze(0)))

requires_grad: False
bate com aligned: True

## 4.5 Congelado x fine-tuning: um passo de gradiente

Para ver a diferenca na pratica, com uma perda de brincadeira
(`saida.pow(2).mean()`):

1. `emb_frozen`: `weight.requires_grad` e `False`, entao chamar `.backward()`
   na saida nem funciona (o `RuntimeError` confirma que nada ali treina).
2. `emb_ft = nn.Embedding.from_pretrained(torch.tensor(aligned), freeze=False,
   padding_idx=0)`: depois de um `backward`, `emb_ft.weight.grad` existe.
   Guarde `w_antes`, faca um passo de SGD manual (`lr=0.1`) e mostre que
   `emb_ft.weight` mudou **so nas linhas usadas** (2, 3, 4) — a linha 0
   continua protegida pelo `padding_idx`.

In [6]:
# 4.5
ids = torch.tensor([[2, 3, 4, 0]])

# congelado: weight.requires_grad = False -> nem da para chamar backward
print("frozen requires_grad:", emb_frozen.weight.requires_grad)
try:
    emb_frozen(ids).pow(2).mean().backward()
except RuntimeError as e:
    print("backward no congelado falha:", str(e)[:55], "...")

# fine-tuning: com gradiente
emb_ft = nn.Embedding.from_pretrained(
    torch.tensor(aligned), freeze=False, padding_idx=0)
w_antes = emb_ft.weight.detach().clone()
emb_ft(ids).pow(2).mean().backward()
with torch.no_grad():
    emb_ft.weight -= 0.1 * emb_ft.weight.grad

mudou = ~torch.isclose(w_antes, emb_ft.weight).all(dim=1)
print("linhas que mudaram:", mudou.nonzero().flatten().tolist())
print("linha 0 (padding) mudou?", bool(mudou[0]))

frozen requires_grad: False
backward no congelado falha: element 0 of tensors does not require grad and does not ...
linhas que mudaram: [2, 3, 4]
linha 0 (padding) mudou? False

## 4.6 Salvar a matriz alinhada

`np.save("aligned_matrix.npy", aligned)` e o `novo_vocab` em
`aligned_vocab.json`. No projeto final e esse par (matriz + vocab na mesma
ordem) que voce carrega no modelo.

In [7]:
# 4.6
np.save("aligned_matrix.npy", aligned)
with open("aligned_vocab.json", "w") as f:
    json.dump({w: i for i, w in enumerate(novo_vocab)}, f)
print("salvo: aligned_matrix.npy, aligned_vocab.json")

salvo: aligned_matrix.npy, aligned_vocab.json

## Resumo

- Matriz pre-treinada so serve se as **linhas seguirem a ordem do `word2idx`**
  do modelo que vai usar; palavras ausentes -> vetor aleatorio pequeno (ou `<UNK>`).
- `from_pretrained(..., freeze=True)`: treina so o resto do modelo (bom com
  pouco dado). `freeze=False`: ajusta os embeddings tambem (precisa de mais dado).
- `padding_idx=0` continua protegendo a linha de padding mesmo em fine-tuning.